<a href="https://colab.research.google.com/github/OmarAyman2005/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import os

REPO_URL = "https://github.com/OmarAyman2005/flyrank-ml-internship.git"
REPO_DIR = "/content/flyrank-ml-internship"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())
print(
    "Starter dataset exists:",
    os.path.exists("data/raw/content_refresh_anonymized.csv")
)

Working directory: /content/flyrank-ml-internship
Starter dataset exists: True


# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarAyman2005/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Lane: Refresh / Content Opportunity Scoring
## Task type: Ranking / scoring

*The goal is to rank content pages by how urgently they should be reviewed for refresh. This is better framed as a ranking/scoring problem than simple classification because the practical decision is not only whether a page may need attention, but which pages should be reviewed first when the team has limited capacity.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*Target / proxy: For the starter dataset, I will use whether a page is currently classified as declining as a proxy for refresh priority. The proxy comes from the observed trend_direction field, where pages marked down are treated as candidates that may deserve earlier review.*

*This is only a starter proxy, not an ideal final target, because it describes the current measurement window rather than a future outcome. A stronger later version would use past signals to predict whether a page declines in a future window.*

In [9]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["refresh_proxy"] = (df["trend_direction"] == "down").astype(int)

print(df["refresh_proxy"].value_counts())

refresh_proxy
1    16262
0    13738
Name: count, dtype: int64


## 3. Success metric

*Success metric: Precision@50*

*Because this is a ranking problem, the most useful question is whether the pages ranked at the very top are actually good refresh candidates. I will therefore use Precision@50: among the top 50 pages recommended for review, how many are truly labeled as declining by the starter proxy. Higher Precision@50 means the limited review capacity is being spent on more relevant pages.*

In [10]:
print("Chosen success metric: Precision@50")
print("Interpretation: fraction of the top 50 ranked pages that are declining.")

Chosen success metric: Precision@50
Interpretation: fraction of the top 50 ranked pages that are declining.


## 4. The unit of analysis, as a real dataframe

*Unit of analysis: one content page per row.*

*Each row represents one anonymized content item/page. The ranking task will use page-level observed signals such as impressions, CTR, average position, content age, and word count to help prioritize which pages should be reviewed first.*

In [11]:
unit_df = df[
    [
        "content_id",
        "impressions_90d",
        "ctr",
        "avg_position",
        "content_age_days",
        "word_count",
        "refresh_proxy"
    ]
].copy()

print("One row = one content page")
print("Shape:", unit_df.shape)

unit_df.head()

One row = one content page
Shape: (30000, 7)


,content_id,impressions_90d,ctr,avg_position,content_age_days,word_count,refresh_proxy
0,content_304f48230142,3803,0.76,10.6,187,3221.0,1
1,content_a1fb4e703a9e,15320,0.05,20.3,445,2481.0,1
2,content_9aa793d4d895,12581,0.09,36.5,141,3515.0,1
3,content_331d6c4de07b,11751,0.49,6.2,463,NaN,0
4,content_d99b7a2d90ca,19140,0.13,44.0,263,2803.0,1


## 5. Why ML beats a fixed rule here

*A fixed rule can only use a small number of hand-chosen conditions, such as “high impressions and declining trend.” In this problem, several signals may matter together — for example impressions, CTR, average position, content age, and word count — and their relationships may not be simple or linear. A learned ranking model can combine these signals and adapt their importance from the data, which may produce a more useful review order than a single fixed rule.*

*The model would still be used only for decision support. An editor or SEO specialist would review the highest-ranked pages before taking any action.*

In [12]:
feature_cols = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "word_count"
]

print("Number of candidate signals:", len(feature_cols))
print("Candidate signals:", feature_cols)

Number of candidate signals: 5
Candidate signals: ['impressions_90d', 'ctr', 'avg_position', 'content_age_days', 'word_count']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.